In [ ]:
!nvidia-smi


In [ ]:
%cd /content
!rm -rf LatentSync
!git clone -q https://github.com/bytedance/LatentSync.git
%cd LatentSync


In [ ]:
!pip install -q -r requirements.txt


In [ ]:
!huggingface-cli download ByteDance/LatentSync-1.5 latentsync_unet.pt --local-dir checkpoints
!huggingface-cli download ByteDance/LatentSync-1.5 whisper/tiny.pt --local-dir checkpoints
!ls -la checkpoints checkpoints/whisper


In [ ]:
!wget -q https://253631672.github.io/macro-bridge-7f3k2/lipsync2/ls_assets.zip -O /content/ls_assets.zip
!mkdir -p /content/assets && cd /content/assets && unzip -o -q /content/ls_assets.zip && ls -la


In [ ]:
import subprocess, os
os.chdir('/content/LatentSync')
for i in range(1, 5):
    out = f'/content/out_s{i}.mp4'
    cmd = ['python', '-m', 'scripts.inference',
           '--unet_config_path', 'configs/unet/stage2.yaml',
           '--inference_ckpt_path', 'checkpoints/latentsync_unet.pt',
           '--inference_steps', '25',
           '--guidance_scale', '1.5',
           '--enable_deepcache',
           '--video_path', f'/content/assets/s{i}.mp4',
           '--audio_path', f'/content/assets/a{i}.wav',
           '--video_out_path', out]
    print('RUN', i, flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True)
    print('rc', r.returncode, 'exists', os.path.exists(out))
    if r.returncode != 0:
        print(r.stdout[-1500:]); print(r.stderr[-1500:])
print('ALL DONE')


In [ ]:
from google.colab import files
import os
for i in range(1, 5):
    p = f'/content/out_s{i}.mp4'
    if os.path.exists(p):
        files.download(p)
print('downloads triggered')
